# Geocodificação híbrida barata e retomável

Este notebook aplica uma estratégia híbrida e barata de geocodificação com cache persistente, checkpoints frequentes e retomada segura após reconexão no Colab.

## Saídas principais

- `estabelecimentos_geocodificados.parquet`
- `qualidade_geocodificacao.csv`
- `geocodificacao_cache.parquet`
- `geocodificacao_estado.json`

In [ ]:
from pathlib import Path
import json

SNAPSHOT_MES = '2026-04'
CIDADE_ALVO = 'Praia Grande'
UF_ALVO = 'SP'
PAIS_ALVO = 'Brasil'

USAR_GOOGLE_DRIVE_PARA_CACHE = True
MONTAR_DRIVE = True
MODO_GEOCODIFICACAO = 'rapido'  # rapido ou completo
TAMANHO_LOTE_GRAVACAO = 25
MAX_CONSULTAS_POR_EXECUCAO = 1000

try:
    from google.colab import drive
except ImportError:
    drive = None

if USAR_GOOGLE_DRIVE_PARA_CACHE and MONTAR_DRIVE and drive is not None:
    drive.mount('/content/drive', force_remount=False)

SNAPSHOT = Path('/content/dados_brutos/cnpj') / SNAPSHOT_MES
PASTA_PREPARADO = SNAPSHOT / 'processado' / 'preparado'
PASTA_PERSISTENTE = (
    Path('/content/drive/MyDrive/geoBusiness/processamento/cnpj') / SNAPSHOT_MES / 'geocodificacao'
    if USAR_GOOGLE_DRIVE_PARA_CACHE
    else PASTA_PREPARADO
)

ESTAB_PREPARADOS = PASTA_PREPARADO / 'estabelecimentos_preparados.parquet'
EMPRESAS_PREPARADAS = PASTA_PREPARADO / 'empresas_preparadas.parquet'
SIMPLES_PREPARADOS = PASTA_PREPARADO / 'simples_preparados.parquet'
QUALIDADE_PREPARACAO = PASTA_PREPARADO / 'qualidade_preparacao.json'

CACHE_GEOCODIFICACAO = PASTA_PERSISTENTE / 'geocodificacao_cache.parquet'
ESTADO_GEOCODIFICACAO = PASTA_PERSISTENTE / 'geocodificacao_estado.json'
SAIDA_GEOCODIFICADA = PASTA_PREPARADO / 'estabelecimentos_geocodificados.parquet'
SAIDA_QUALIDADE = PASTA_PREPARADO / 'qualidade_geocodificacao.csv'
SAIDA_GEOCODIFICADA_PERSISTENTE = PASTA_PERSISTENTE / 'estabelecimentos_geocodificados.parquet'
SAIDA_QUALIDADE_PERSISTENTE = PASTA_PERSISTENTE / 'qualidade_geocodificacao.csv'

PASTA_PREPARADO.mkdir(parents=True, exist_ok=True)
PASTA_PERSISTENTE.mkdir(parents=True, exist_ok=True)

ARQUIVOS_ESPERADOS_PREPARADO = {
    'estabelecimentos_preparados': ESTAB_PREPARADOS,
    'empresas_preparadas': EMPRESAS_PREPARADAS,
    'simples_preparados': SIMPLES_PREPARADOS,
    'qualidade_preparacao': QUALIDADE_PREPARACAO,
}

print('Estrutura pronta para upload manual do preparado:')
print(f'- pasta de entrada local: {PASTA_PREPARADO}')
print(f'- pasta persistente: {PASTA_PERSISTENTE}')
print(f'- modo de geocodificação: {MODO_GEOCODIFICACAO}')
print(f'- consultas máximas por execução: {MAX_CONSULTAS_POR_EXECUCAO}')
print('Arquivos esperados em preparado:')
for nome, caminho in ARQUIVOS_ESPERADOS_PREPARADO.items():
    print(f'  - {nome}: {caminho.name}')

print('Arquivos persistentes do notebook 04:')
print(f'  - cache: {CACHE_GEOCODIFICACAO}')
print(f'  - estado: {ESTADO_GEOCODIFICACAO}')
print(f'  - saída principal: {SAIDA_GEOCODIFICADA_PERSISTENTE}')
print(f'  - qualidade: {SAIDA_QUALIDADE_PERSISTENTE}')

ARQUIVOS_ESPERADOS_PREPARADO


In [ ]:
import numpy as np
import pandas as pd

try:
    import pyarrow  # noqa: F401
except ImportError:
    !pip -q install pyarrow
    import pyarrow  # noqa: F401

try:
    from geopy.extra.rate_limiter import RateLimiter
    from geopy.geocoders import Nominatim
except ImportError:
    !pip -q install geopy
    from geopy.extra.rate_limiter import RateLimiter
    from geopy.geocoders import Nominatim

if not ESTAB_PREPARADOS.exists():
    raise FileNotFoundError(f'Arquivo preparado ausente: {ESTAB_PREPARADOS}')

PAUSA_NOMINATIM_SEGUNDOS = 1.1
ESTAB_PREPARADOS


In [ ]:
estabelecimentos = pd.read_parquet(ESTAB_PREPARADOS)
estabelecimentos.shape


In [ ]:
def limpar_texto(valor: object) -> str:
    if valor is None:
        return ''
    return ' '.join(str(valor).strip().split())

def consulta_endereco_completo(linha: pd.Series) -> str:
    partes = [
        limpar_texto(linha.get('logradouro_limpo')),
        limpar_texto(linha.get('numero_limpo')),
        limpar_texto(linha.get('bairro_limpo')),
        CIDADE_ALVO,
        UF_ALVO,
        PAIS_ALVO,
    ]
    partes = [parte for parte in partes if parte]
    return ', '.join(partes)

def consulta_por_cep(cep: str) -> str:
    cep = limpar_texto(cep)
    if len(cep) != 8:
        return ''
    return ', '.join([cep, CIDADE_ALVO, UF_ALVO, PAIS_ALVO])

def consulta_bairro(bairro: str) -> str:
    bairro = limpar_texto(bairro)
    if not bairro:
        return ''
    return ', '.join([bairro, CIDADE_ALVO, UF_ALVO, PAIS_ALVO])

def persistir_cache(cache_df: pd.DataFrame, estado: dict) -> None:
    cache_df.to_parquet(CACHE_GEOCODIFICACAO, index=False)
    with ESTADO_GEOCODIFICACAO.open('w', encoding='utf-8') as arquivo:
        json.dump(estado, arquivo, ensure_ascii=False, indent=2, default=str)

def carregar_estado() -> dict:
    if ESTADO_GEOCODIFICACAO.exists():
        return json.loads(ESTADO_GEOCODIFICACAO.read_text(encoding='utf-8'))
    return {}


In [ ]:
enderecos = estabelecimentos[['chave_endereco', 'logradouro_limpo', 'numero_limpo', 'bairro_limpo', 'cep_limpo']].drop_duplicates().copy()
consultas = []

if MODO_GEOCODIFICACAO == 'completo':
    enderecos_completos = enderecos.loc[
        enderecos['logradouro_limpo'].fillna('').astype(str).str.strip().ne('')
        & enderecos['numero_limpo'].fillna('').astype(str).str.strip().ne('')
        & enderecos['bairro_limpo'].fillna('').astype(str).str.strip().ne('')
    ].copy()
    enderecos_completos['consulta_geocodificacao'] = enderecos_completos.apply(consulta_endereco_completo, axis=1)
    for _, linha in enderecos_completos.iterrows():
        consultas.append({
            'tipo_chave': 'chave_endereco',
            'valor_chave': linha['chave_endereco'],
            'consulta_geocodificacao': linha['consulta_geocodificacao'],
            'metodo_planejado': 'endereco_completo',
            'confianca_planejada': 'alta',
            'ordem': 1,
        })

ceps = enderecos['cep_limpo'].fillna('').astype(str).str.strip()
for cep in sorted(valor for valor in ceps.unique() if len(valor) == 8):
    consultas.append({
        'tipo_chave': 'cep_limpo',
        'valor_chave': cep,
        'consulta_geocodificacao': consulta_por_cep(cep),
        'metodo_planejado': 'cep',
        'confianca_planejada': 'media',
        'ordem': 2,
    })

for bairro in sorted(valor for valor in enderecos['bairro_limpo'].fillna('').astype(str).str.strip().unique() if valor):
    consultas.append({
        'tipo_chave': 'bairro_limpo',
        'valor_chave': bairro,
        'consulta_geocodificacao': consulta_bairro(bairro),
        'metodo_planejado': 'bairro',
        'confianca_planejada': 'baixa',
        'ordem': 3,
    })

consultas = pd.DataFrame(consultas).drop_duplicates()
consultas.groupby('metodo_planejado').size().reset_index(name='quantidade')


In [ ]:
colunas_cache = ['consulta_geocodificacao', 'metodo_planejado', 'latitude', 'longitude', 'endereco_encontrado', 'sucesso_consulta', 'timestamp_consulta']
if CACHE_GEOCODIFICACAO.exists():
    cache = pd.read_parquet(CACHE_GEOCODIFICACAO)
else:
    cache = pd.DataFrame(columns=colunas_cache)

consultas_unicas = consultas[['tipo_chave', 'valor_chave', 'consulta_geocodificacao', 'metodo_planejado', 'confianca_planejada', 'ordem']].drop_duplicates()
pendentes = consultas_unicas.merge(
    cache[['consulta_geocodificacao', 'metodo_planejado']].drop_duplicates(),
    on=['consulta_geocodificacao', 'metodo_planejado'],
    how='left',
    indicator=True,
)
pendentes = pendentes.loc[pendentes['_merge'] == 'left_only', ['tipo_chave', 'valor_chave', 'consulta_geocodificacao', 'metodo_planejado', 'confianca_planejada', 'ordem']]
pendentes = pendentes.sort_values(['ordem', 'tipo_chave', 'valor_chave'])

if MAX_CONSULTAS_POR_EXECUCAO is not None:
    pendentes = pendentes.head(MAX_CONSULTAS_POR_EXECUCAO)

estado_atual = carregar_estado()
print(f'Consultas pendentes nesta execução: {len(pendentes)}')
estado_atual


In [ ]:
geolocator = Nominatim(user_agent='geobusiness-poc')
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=PAUSA_NOMINATIM_SEGUNDOS, swallow_exceptions=True)

novos_registros = []
processadas_nesta_execucao = 0

for indice, linha in pendentes.reset_index(drop=True).iterrows():
    consulta = linha['consulta_geocodificacao']
    metodo = linha['metodo_planejado']
    local = geocode(consulta, country_codes='br', addressdetails=True, exactly_one=True, timeout=15)

    novos_registros.append({
        'consulta_geocodificacao': consulta,
        'metodo_planejado': metodo,
        'latitude': np.nan if local is None else float(local.latitude),
        'longitude': np.nan if local is None else float(local.longitude),
        'endereco_encontrado': '' if local is None else str(local.address),
        'sucesso_consulta': bool(local is not None),
        'timestamp_consulta': pd.Timestamp.utcnow(),
    })
    processadas_nesta_execucao += 1

    if (indice + 1) % TAMANHO_LOTE_GRAVACAO == 0:
        cache = pd.concat([cache, pd.DataFrame(novos_registros)], ignore_index=True)
        cache = cache.drop_duplicates(subset=['consulta_geocodificacao', 'metodo_planejado'], keep='last')
        estado = {
            'snapshot_referencia': SNAPSHOT_MES,
            'modo_geocodificacao': MODO_GEOCODIFICACAO,
            'consultas_processadas_nesta_execucao': processadas_nesta_execucao,
            'consultas_totais_em_cache': int(len(cache)),
            'consultas_pendentes_restantes': int(max(len(pendentes) - processadas_nesta_execucao, 0)),
            'atualizado_em': str(pd.Timestamp.utcnow()),
        }
        persistir_cache(cache, estado)
        novos_registros = []
        print(f'Checkpoint salvo: {processadas_nesta_execucao} consultas nesta execução')

if novos_registros:
    cache = pd.concat([cache, pd.DataFrame(novos_registros)], ignore_index=True)
    cache = cache.drop_duplicates(subset=['consulta_geocodificacao', 'metodo_planejado'], keep='last')

estado_final = {
    'snapshot_referencia': SNAPSHOT_MES,
    'modo_geocodificacao': MODO_GEOCODIFICACAO,
    'consultas_processadas_nesta_execucao': processadas_nesta_execucao,
    'consultas_totais_em_cache': int(len(cache)),
    'consultas_pendentes_restantes': int(max(len(pendentes) - processadas_nesta_execucao, 0)),
    'atualizado_em': str(pd.Timestamp.utcnow()),
}
persistir_cache(cache, estado_final)
estado_final


In [ ]:
resolucao = consultas.merge(cache, on=['consulta_geocodificacao', 'metodo_planejado'], how='left')
resolucao = resolucao.sort_values(['ordem', 'tipo_chave', 'valor_chave'])
resolucao_sucesso = resolucao.loc[resolucao['sucesso_consulta'] == True].copy()

def preparar_resolucao(tipo_chave: str, nome_merge: str) -> pd.DataFrame:
    base = resolucao_sucesso.loc[resolucao_sucesso['tipo_chave'] == tipo_chave, ['valor_chave', 'metodo_planejado', 'confianca_planejada', 'latitude', 'longitude', 'endereco_encontrado']].drop_duplicates('valor_chave')
    if base.empty:
        return base
    return base.rename(columns={
        'valor_chave': tipo_chave,
        'metodo_planejado': f'metodo_{nome_merge}',
        'confianca_planejada': f'confianca_{nome_merge}',
        'latitude': f'latitude_{nome_merge}',
        'longitude': f'longitude_{nome_merge}',
        'endereco_encontrado': f'endereco_{nome_merge}',
    })

estabelecimentos_geocodificados = estabelecimentos.copy()

for tipo_chave, nome_merge in [('chave_endereco', 'endereco'), ('cep_limpo', 'cep'), ('bairro_limpo', 'bairro')]:
    resolvido = preparar_resolucao(tipo_chave, nome_merge)
    if not resolvido.empty:
        estabelecimentos_geocodificados = estabelecimentos_geocodificados.merge(resolvido, on=tipo_chave, how='left')

estabelecimentos_geocodificados['metodo_geocodificacao'] = pd.NA
estabelecimentos_geocodificados['confianca_geografica'] = pd.NA
estabelecimentos_geocodificados['latitude_final'] = pd.NA
estabelecimentos_geocodificados['longitude_final'] = pd.NA
estabelecimentos_geocodificados['endereco_geocodificado'] = pd.NA

for origem in ['endereco', 'cep', 'bairro']:
    metodo_col = f'metodo_{origem}'
    confianca_col = f'confianca_{origem}'
    latitude_col = f'latitude_{origem}'
    longitude_col = f'longitude_{origem}'
    endereco_col = f'endereco_{origem}'
    if metodo_col in estabelecimentos_geocodificados.columns:
        estabelecimentos_geocodificados['metodo_geocodificacao'] = estabelecimentos_geocodificados['metodo_geocodificacao'].fillna(estabelecimentos_geocodificados[metodo_col])
        estabelecimentos_geocodificados['confianca_geografica'] = estabelecimentos_geocodificados['confianca_geografica'].fillna(estabelecimentos_geocodificados[confianca_col])
        estabelecimentos_geocodificados['latitude_final'] = estabelecimentos_geocodificados['latitude_final'].fillna(estabelecimentos_geocodificados[latitude_col])
        estabelecimentos_geocodificados['longitude_final'] = estabelecimentos_geocodificados['longitude_final'].fillna(estabelecimentos_geocodificados[longitude_col])
        estabelecimentos_geocodificados['endereco_geocodificado'] = estabelecimentos_geocodificados['endereco_geocodificado'].fillna(estabelecimentos_geocodificados[endereco_col])

estabelecimentos_geocodificados['metodo_geocodificacao'] = estabelecimentos_geocodificados['metodo_geocodificacao'].fillna('nao_localizado')
estabelecimentos_geocodificados['confianca_geografica'] = estabelecimentos_geocodificados['confianca_geografica'].fillna('baixa')
estabelecimentos_geocodificados['precisa_revisao_geocodificacao'] = estabelecimentos_geocodificados['confianca_geografica'] != 'alta'
estabelecimentos_geocodificados['latitude'] = estabelecimentos_geocodificados['latitude_final']
estabelecimentos_geocodificados['longitude'] = estabelecimentos_geocodificados['longitude_final']

estabelecimentos_geocodificados.to_parquet(SAIDA_GEOCODIFICADA, index=False)
if SAIDA_GEOCODIFICADA_PERSISTENTE != SAIDA_GEOCODIFICADA:
    estabelecimentos_geocodificados.to_parquet(SAIDA_GEOCODIFICADA_PERSISTENTE, index=False)

qualidade = estabelecimentos_geocodificados.groupby(['confianca_geografica', 'metodo_geocodificacao'], dropna=False).size().reset_index(name='quantidade')
qualidade['percentual'] = qualidade['quantidade'] / max(len(estabelecimentos_geocodificados), 1)
qualidade.to_csv(SAIDA_QUALIDADE, index=False)
if SAIDA_QUALIDADE_PERSISTENTE != SAIDA_QUALIDADE:
    qualidade.to_csv(SAIDA_QUALIDADE_PERSISTENTE, index=False)

qualidade
